
# EMAIL (v3) — Release-aware Viewer (no writes)

Zero guesswork. This inspects raw `email.csv` for the current release, proves what exists, dry-runs Send/Receive dedupe when relevant, verifies the `@dtaa` assumption, and previews lean/full target columns.  
It **does not** write Parquets.


In [ ]:

# %% bootstrap
from nb_paths import bootstrap, csv_path, read_csv
import pandas as pd, numpy as np, re
from pathlib import Path

env = bootstrap()
EMAIL_PATH = csv_path(env, "email")
print(f"RELEASE={env.RELEASE}  |  EMAIL={EMAIL_PATH}")


In [ ]:

# %% raw header + dtypes (exact, no aliasing)
head = read_csv(env, "email", nrows=10)
print("Columns (raw):", list(head.columns))
print("\nDtypes (raw):")
print(head.dtypes)
head


In [ ]:

# %% helpers (no guessing, CERT canonical names only)

# Canonical column names that CERT actually uses across releases.
# Some may be missing in a given release; that's fine.
CANON = {
    "id", "date", "from", "to", "cc", "bcc",
    "user", "pc", "activity", "attachments", "size",
    "content",
}

def have(df, name: str):
    return name if name in df.columns else None

def get(df, name: str):
    if name in df.columns:
        return df[name]
    import pandas as pd
    return pd.Series([pd.NA] * len(df), dtype="object")

# Recipient parsing (lightweight, no explosion)
RECIP_SPLIT = re.compile(r"[,\s;|]+")

def split_recipients(raw):
    if raw is None: return []
    s = str(raw).strip()
    if not s: return []
    return [p for p in RECIP_SPLIT.split(s) if p]

# Email domain extraction
def extract_domain(addr: str):
    if not isinstance(addr, str): return None
    m = re.search(r"@([\w\.-]+)", addr.strip().lower())
    return m.group(1) if m else None

# Month floor
def to_month(ts):
    try:
        return pd.to_datetime(ts).to_period("M").to_timestamp(how="start")
    except Exception:
        return pd.NaT

# Fixed internal domain set (validated later)
INTERNAL_DOMAINS = {"dtaa.com"}


In [ ]:

# %% load full file as strings (no schema coercion)
df_raw = read_csv(env, "email", dtype=str)
print("Rows:", len(df_raw))
list(df_raw.columns)[:25]


In [ ]:

# %% presence map for canonical fields
present = {name: (name in df_raw.columns) for name in sorted(CANON)}
present


In [ ]:

# %% ID sanity
id_col = have(df_raw, "id")
act_col = have(df_raw, "activity")
if id_col is None:
    print("No 'id' column present. Skipping id-based summaries.")
    id_dups = pd.DataFrame()
else:
    counts = df_raw[id_col].value_counts(dropna=False)
    dup = counts[counts > 1]
    print(f"Distinct ids: {counts.size}  |  duplicate ids: {dup.size}  |  max dup group size: {int(dup.max()) if not dup.empty else 1}")
    if act_col is not None:
        id_dups = (df_raw[df_raw[id_col].isin(dup.index)]
                   .groupby(id_col)[act_col].agg(lambda s: list(pd.Series(s).astype(str).str.lower()))
                   .reset_index().rename(columns={act_col: "activities"}))
        id_dups.head(10)
    else:
        id_dups = (df_raw[df_raw[id_col].isin(dup.index)]
                   .groupby(id_col).size().reset_index(name="rows_in_group")).head(10)
        id_dups


In [ ]:

# %% activity inventory
if act_col is None:
    print("No 'activity' column present in this release (expected for r3.1).")
else:
    act_norm = df_raw[act_col].astype(str).str.strip().str.lower()
    print(act_norm.value_counts(dropna=False).head(20))
    # sample 5 rows per value
    samples = []
    for val in list(act_norm.value_counts().index)[:10]:
        ex = df_raw[act_norm == val].head(5)
        ex = ex[[c for c in [have(df_raw,'id'), act_col, have(df_raw,'from'), have(df_raw,'to')] if c is not None]]
        samples.append(ex)
    pd.concat(samples, ignore_index=True) if samples else pd.DataFrame()


In [ ]:
# %% Send/Receive dedupe DRY-RUN (fast path)
will_dedupe = False
drop_candidate_rows = 0
if (id_col is not None) and (act_col is not None):
    if not df_raw[id_col].is_unique:
        act = df_raw[act_col].astype("string").str.lower().str.strip()
        ids = df_raw[id_col]
        has_send = act.eq("send").groupby(ids, sort=False).any()
        has_recv = act.eq("receive").groupby(ids, sort=False).any()
        ids_with_both = has_send & has_recv
        drop_candidate_rows = int(
            act.eq("receive").groupby(ids, sort=False).sum()[ids_with_both].sum()
        )
        will_dedupe = drop_candidate_rows > 0
else:
    print("Dedupe preview skipped (need both 'id' and 'activity').")

print(f"Will dedupe (if we were emitting): {will_dedupe}  |  receive rows that would drop: {drop_candidate_rows}")

In [ ]:

# %% timestamp + event_month probe
date_col = have(df_raw, "date")
if date_col is None:
    print("No 'date' column present; cannot compute timestamp/event_month preview.")
    df_ts = pd.DataFrame()
else:
    ts = pd.to_datetime(df_raw[date_col], errors="coerce")
    em = ts.dt.to_period('M').dt.to_timestamp(how='start')
    print("Parsed timestamp nulls:", int(ts.isna().sum()))
    print("Event months:", int(em.nunique()))
    df_ts = pd.DataFrame({"timestamp": ts, "event_month": em}).head(10)
df_ts


In [ ]:

# %% domain check (prove '@dtaa')
from_col = have(df_raw, "from")
to_col   = have(df_raw, "to")
cc_col   = have(df_raw, "cc")
bcc_col  = have(df_raw, "bcc")

# sender domains
if from_col is not None:
    from_domains = df_raw[from_col].astype(str).map(extract_domain)
    print("Top 20 sender domains:")
    print(from_domains.value_counts(dropna=False).head(20))
else:
    print("No 'from' column; cannot summarize sender domains.")

# recipient domains (sampled to avoid RAM pain)
def collect_domains(col):
    if col is None: return []
    s = df_raw[col].head(10000)
    out = []
    for raw in s:
        for token in split_recipients(raw):
            d = extract_domain(token)
            if d: out.append(d)
    return out

recip_domains = collect_domains(to_col) + collect_domains(cc_col) + collect_domains(bcc_col)
recip_series = pd.Series(recip_domains, dtype=object)
print("\nTop 20 recipient domains (sampled):")
print(recip_series.value_counts(dropna=False).head(20))

has_dtaa_sender = (from_col is not None) and ((from_domains == "dtaa.com").sum() > 0) if from_col is not None else False
has_dtaa_recip  = (recip_series == "dtaa.com").sum() > 0
print(f"\n'@dtaa' appears in senders: {bool(has_dtaa_sender)}  |  recipients(sample): {bool(has_dtaa_recip)}")


In [ ]:
# %% FAST preview of external-recipient flag
TO = have(df_raw, "to")
if TO is None:
    print("No 'to' column; skipping.")
else:
    to_s = df_raw[TO].fillna("").astype(str).str.lower()

    # Count a row as external if any address doesn't end with dtaa.com or .dtaa.com
    any_external = to_s.str.contains(
        r'@(?!(?:[\w\.-]*\.)?dtaa\.com\b)[\w\.-]+\b',
        flags=re.IGNORECASE, regex=True, na=False
    )

    # optional preview on first 200k to keep runtime small
    N = min(200_000, len(to_s))
    print("Preview (first", N, "rows) — any external recipient:")
    print(any_external.iloc[:N].value_counts())

In [ ]:
# %% email_to_ext_domain (full run, still fast)
TO = have(df_raw, "to")
if TO is None:
    print("email_to_ext_domain: [skipped] — no 'to' column in this release.")
else:
    to_s = df_raw[TO].fillna("").astype(str).str.lower()
    email_to_ext_domain = to_s.str.contains(
        r'@(?!(?:[\w\.-]*\.)?dtaa\.com\b)[\w\.-]+\b',
        flags=re.IGNORECASE, regex=True, na=False
    )
    print("email_to_ext_domain counts (full):")
    print(email_to_ext_domain.value_counts())

In [ ]:
# %% sender_is_internal_email counts
FR = have(df_raw, "from")
if FR is None:
    print("sender_is_internal_email: [skipped] — no 'from' column in this release.")
else:
    sender_internal = df_raw[FR].fillna("").astype(str).str.lower().str.endswith("dtaa.com")
    print("sender_is_internal_email counts (full):")
    print(sender_internal.value_counts())

In [ ]:

# %% attachments + size reconnaissance
att_col = have(df_raw, "attachments")
size_col = have(df_raw, "size")

def preview_series(colname, n=10):
    if colname is None:
        print("  [missing]")
        return
    s = df_raw[colname]
    print("  top values:")
    print(s.value_counts(dropna=False).head(n))

print("attachments (raw)")
preview_series(att_col)
print("\nsize (raw)")
preview_series(size_col)

def parse_attachment_count(val):
    if val is None or (isinstance(val, float) and pd.isna(val)): 
        return np.nan
    s = str(val).strip()
    if re.fullmatch(r"\d+", s):
        try: return int(s)
        except Exception: return np.nan
    tokens = [t for t in re.split(r"[;,\|\s]+", s) if t]
    if len(tokens) == 1 and "." not in tokens[0]:
        return np.nan
    return len(tokens) if tokens else np.nan

if att_col is not None:
    att_head = df_raw[att_col].head(2000)
    att_count_preview = att_head.map(parse_attachment_count)
    print("\nattachment_count preview (first 2000 rows):")
    print(att_count_preview.value_counts(dropna=False).head(10))

if size_col is not None:
    size_head = pd.to_numeric(df_raw[size_col].head(20000), errors="coerce")
    print("\nsize numeric cast — nulls after cast (first 20k):", int(size_head.isna().sum()))


In [ ]:

# %% recipient counts preview (to/cc/bcc)
def count_tokens(raw):
    return len(split_recipients(raw)) if isinstance(raw, str) else 0

for key in ["to","cc","bcc"]:
    col = have(df_raw, key)
    if col is not None:
        s = df_raw[col].head(20000).map(count_tokens)
        print(key+"_count →", s.describe())
    else:
        print(key+"_count → [missing]")


In [ ]:
# %% LDAP joinability preview (sender only, lowercased email + month) — no src import
from pathlib import Path

from_col = have(df_raw, "from")
date_col = have(df_raw, "date")

if (from_col is None) or (date_col is None):
    print("Cannot preview LDAP joinability (need 'from' and 'date').")
else:
    sender_email_l = df_raw[from_col].astype(str).str.strip().str.lower()
    em = pd.to_datetime(df_raw[date_col], errors="coerce").dt.to_period('M').dt.to_timestamp(how='start')

    base = Path(env.OUT) / env.RELEASE
    lean = base / "ldap_v3_lean" / "ldap_asof_by_month.parquet"
    full = base / "ldap_v3_full" / "ldap_asof_by_month.parquet"
    asof_path = lean if lean.exists() else full

    print("Using LDAP as-of:", asof_path, "| exists:", asof_path.exists())
    if asof_path.exists():
        ldap = pd.read_parquet(asof_path, columns=["email","event_month"])
        ldap["email"] = ldap["email"].astype(str).str.strip().str.lower()
        ldap["event_month"] = pd.to_datetime(ldap["event_month"]).dt.to_period('M').dt.to_timestamp(how='start')

        N = min(200_000, len(df_raw))
        df_samp = pd.DataFrame({
            "sender_email_l": sender_email_l.iloc[:N],
            "event_month": em.iloc[:N]
        })
        j = df_samp.merge(
            ldap.rename(columns={"email": "ldap_email"}),
            left_on=["sender_email_l","event_month"],
            right_on=["ldap_email","event_month"],
            how="left"
        )
        rate = float(j["ldap_email"].notna().mean()) if len(j) else 0.0
        print(f"Preview join rate (first {N:,} rows): {rate:.2%}")
    else:
        print("Build LDAP first to preview joinability.")

In [ ]:

# %% preview: final column sets (no writes)
LEAN_COLS = [
    "timestamp","event_month","id",
    "user_key","pc","activity",
    "size","attachment_count","to_count","cc_count","bcc_count",
    "email_to_ext_domain",
    "role","is_admin","team","supervisor_key",
    "after_hours","user_is_active_employee",
]
FULL_COLS = [
    "timestamp","event_month","id","date",
    "user_key","user_raw","pc","activity",
    "from","to","cc","bcc","content",
    "size","attachment_count","to_count","cc_count","bcc_count",
    "from_domain","sent_from_personal_email","email_to_ext_domain",
    "role","is_admin","team","supervisor_key","first_seen","last_seen",
    "after_hours","user_is_active_employee",
    "joined_ldap","join_reason",
]
print("LEAN target columns:", LEAN_COLS)
print("FULL  target columns:", FULL_COLS)


In [ ]:

# %% QC summary (clipboard friendly)
print("=== QC SUMMARY ===")
if act_col is not None:
    print("Activity values:", df_raw[act_col].astype(str).str.lower().value_counts(dropna=False).to_dict())
else:
    print("Activity values: [missing]")

if id_col is not None:
    vc = df_raw[id_col].value_counts(dropna=False)
    print(f"ID duplicates: {int((vc>1).sum())}  |  Rows that would drop (receive w/ matching send):", int(drop_candidate_rows))
else:
    print("ID column: [missing]")

print("Internal domain fixed set: {'dtaa'} (verified above via sender/recipient domain lists)")
print("Note: This viewer does not write any artifacts. Use results to adjust ETL emitters.")


In [ ]:
# === EMAIL PARQUET SANITY CHECKS (ONLY RUN after parquets have been produced) ===
# Goal: be lightweight. We use parquet metadata (cheap) + tiny DuckDB samples.
# Outputs: human-readable prints + a compact JSON summary for your PR.

from pathlib import Path
import json
import os
import math

# nb_paths is read-only helpers
try:
    from notebooks.nb_paths import bootstrap
except Exception:
    from nb_paths import bootstrap

env = bootstrap()
base = Path(env.OUT) / env.RELEASE / "email_v3"
full_p = base / "email_v3_full" / "email_full.parquet"
lean_p = base / "email_v3_lean" / "email_lean.parquet"
edges_p = base / "email_edges.parquet"

# ---------- helpers: ultra-cheap metadata peeks ----------
try:
    import pyarrow.parquet as pq
except Exception as e:
    raise RuntimeError("pyarrow is required for this sanity cell") from e

def pq_exists(path: Path) -> bool:
    return path.exists() and path.is_file()

def pq_meta(path: Path):
    if not pq_exists(path):
        return dict(exists=False, rows=0, cols=[], size_bytes=0)
    try:
        pf = pq.ParquetFile(str(path))
        meta = getattr(pf, "metadata", None)
        rows = int(meta.num_rows) if meta is not None else 0
        cols = list(pf.schema.names)
        size = path.stat().st_size
        return dict(exists=True, rows=rows, cols=cols, size_bytes=size)
    except Exception as e:
        return dict(exists=True, rows=0, cols=[], size_bytes=path.stat().st_size, error=str(e))

def fmt_mb(n): 
    return f"{n/1024/1024:.1f} MiB"

m_full = pq_meta(full_p)
m_lean = pq_meta(lean_p)
m_edges = pq_meta(edges_p)

print("== EMAIL parquet presence + metadata ==")
for name, m, p in [
    ("email_full", m_full, full_p),
    ("email_lean", m_lean, lean_p),
    ("email_edges", m_edges, edges_p),
]:
    print(f"{name:12} exists={m['exists']} rows={m['rows']:,} cols={len(m['cols'])} size≈{fmt_mb(m['size_bytes'])} path={p}")

# ---------- tiny-sample stats with DuckDB (super cheap) ----------
# Notes:
# - Uses Bernoulli TABLESAMPLE for ~0.1% unless you override EMAIL_SAMPLE_PCT.
# - All queries guarded by column-existence checks.
sample_pct = float(os.getenv("EMAIL_SAMPLE_PCT", "0.1"))  # percent
sample_pct = max(0.01, min(sample_pct, 1.0))              # clamp to [0.01, 1.0]

try:
    import duckdb
except Exception:
    duckdb = None
    print("\n[warn] DuckDB not installed; skipping sample-based checks.")

summary = {
    "release": env.RELEASE,
    "family": "email_v3",
    "artifacts": {
        "full": {"path": str(full_p), **m_full},
        "lean": {"path": str(lean_p), **m_lean},
        "edges": {"path": str(edges_p), **m_edges},
    },
    "checks": {}
}

def has_col(con, tname, col):
    return bool(con.execute(f"SELECT COUNT(*) FROM pragma_table_info('{tname}') WHERE name='{col}'").fetchone()[0])

if duckdb and pq_exists(lean_p):
    print(f"\n== EMAIL LEAN sampled checks (≈ {sample_pct}% sample) ==")
    con = duckdb.connect(database=":memory:")
    con.execute(f"CREATE OR REPLACE TABLE el AS SELECT * FROM read_parquet('{lean_p}') TABLESAMPLE BERNOULLI({sample_pct})")
    # min/max time if present
    time_cols = [c for c in ("timestamp","event_month") if has_col(con, "el", c)]
    stats = {}
    if "timestamp" in time_cols:
        tmin, tmax = con.execute("SELECT MIN(timestamp), MAX(timestamp) FROM el").fetchone()
        print(f"timestamp range: {tmin} .. {tmax}")
        stats["timestamp_min"] = str(tmin)
        stats["timestamp_max"] = str(tmax)
    if "event_month" in time_cols:
        mmin, mmax = con.execute("SELECT MIN(event_month), MAX(event_month) FROM el").fetchone()
        print(f"event_month range: {mmin} .. {mmax}")
        stats["event_month_min"] = str(mmin)
        stats["event_month_max"] = str(mmax)

    # null-rate on key columns if present
    key_cols = [c for c in ("id","from_key","to_key","size","attachment_count") if has_col(con, "el", c)]
    if key_cols:
        q = " UNION ALL ".join([
            f"SELECT '{c}' AS col, ROUND(100.0*AVG(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END),2) AS pct_null FROM el" 
            for c in key_cols
        ])
        rows = con.execute(q).fetchall()
        print("null-rate (% on sample):")
        for col, pct in rows:
            print(f"  {col:16} {pct:6.2f}%")
        stats["null_rate_pct"] = {col: float(pct) for col, pct in rows}
    # quick sanity of from_key cardinality
    if has_col(con, "el", "from_key"):
        distinct_from = con.execute("SELECT COUNT(DISTINCT from_key) FROM el WHERE from_key IS NOT NULL").fetchone()[0]
        total = con.execute("SELECT COUNT(*) FROM el").fetchone()[0]
        print(f"distinct senders (sample): {int(distinct_from):,} out of {int(total):,} rows")
        stats["distinct_from_key_sample"] = int(distinct_from)
        stats["sample_rows"] = int(total)

    summary["checks"]["lean_sample"] = stats

if duckdb and pq_exists(full_p):
    print(f"\n== EMAIL FULL sampled checks (≈ {sample_pct}% sample) ==")
    con2 = duckdb.connect(database=":memory:")
    con2.execute(f"CREATE OR REPLACE TABLE ef AS SELECT * FROM read_parquet('{full_p}') TABLESAMPLE BERNOULLI({sample_pct})")
    stats_f = {}
    # content presence if present
    if has_col(con2, "ef", "content"):
        present = con2.execute("SELECT ROUND(100.0*AVG(CASE WHEN content IS NULL OR content='' THEN 0 ELSE 1 END),2) FROM ef").fetchone()[0]
        print(f"content present (non-empty) on sample: {present:.2f}%")
        stats_f["content_present_pct"] = float(present)
    # attachments distribution if present
    if has_col(con2, "ef", "attachment_count"):
        bins = con2.execute("""
            SELECT 
              CASE
                WHEN attachment_count IS NULL THEN 'NULL'
                WHEN attachment_count=0 THEN '0'
                WHEN attachment_count=1 THEN '1'
                WHEN attachment_count BETWEEN 2 AND 5 THEN '2-5'
                ELSE '6+'
              END AS bin, COUNT(*) AS c
            FROM ef GROUP BY 1 ORDER BY 
              CASE bin WHEN 'NULL' THEN 0 WHEN '0' THEN 1 WHEN '1' THEN 2 WHEN '2-5' THEN 3 ELSE 4 END
        """).fetchall()
        print("attachment_count bins (sample):")
        for b, c in bins:
            print(f"  {b:5} {int(c):,}")
        stats_f["attachment_bins_sample"] = [{ "bin": b, "count": int(c)} for b, c in bins]
    summary["checks"]["full_sample"] = stats_f

# Optional: edges sanity
if duckdb and pq_exists(edges_p):
    print(f"\n== EMAIL EDGES sampled checks (≈ {sample_pct}% sample) ==")
    con3 = duckdb.connect(database=":memory:")
    con3.execute(f"CREATE OR REPLACE TABLE ee AS SELECT * FROM read_parquet('{edges_p}') TABLESAMPLE BERNOULLI({sample_pct})")
    # verify expected columns exist
    cols_ok = all(
        has_col(con3, "ee", c) for c in ("id","from_key","to_key","timestamp","event_month")
        if pq_exists(edges_p)
    )
    print(f"edges columns OK on sample: {cols_ok}")
    summary["checks"]["edges_sample"] = {"columns_ok": bool(cols_ok)}

# ---------- write compact JSON sidecar for PRs ----------
qc_dir = Path(env.OUT) / env.RELEASE / "qc"
qc_dir.mkdir(parents=True, exist_ok=True)
out_json = qc_dir / "email_parquet_sanity.json"
out_json.write_text(json.dumps(summary, indent=2, default=str))
print(f"\nWrote {out_json}")